In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Facial Emotion Detection - Data Exploration\n",
    "\n",
    "This notebook provides exploratory data analysis of the FER-2013 dataset and model experimentation."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "import os\n",
    "sys.path.append('../src')\n",
    "\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from data_preprocessing import DataPreprocessor\n",
    "from model import EmotionCNN\n",
    "import cv2\n",
    "\n",
    "# Set style\n",
    "plt.style.use('seaborn-v0_8')\n",
    "sns.set_palette(\"husl\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Load and Explore Dataset"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load FER-2013 dataset\n",
    "df = pd.read_csv('../data/fer2013.csv')\n",
    "print(f\"Dataset shape: {df.shape}\")\n",
    "print(f\"Columns: {df.columns.tolist()}\")\n",
    "print(\"\\nFirst few rows:\")\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Emotion distribution\n",
    "emotion_labels = {\n",
    "    0: 'Angry', 1: 'Disgust', 2: 'Fear', 3: 'Happy',\n",
    "    4: 'Sad', 5: 'Surprise', 6: 'Neutral'\n",
    "}\n",
    "\n",
    "df['emotion_name'] = df['emotion'].map(emotion_labels)\n",
    "\n",
    "plt.figure(figsize=(12, 6))\n",
    "\n",
    "# Count plot\n",
    "plt.subplot(1, 2, 1)\n",
    "emotion_counts = df['emotion_name'].value_counts()\n",
    "bars = plt.bar(emotion_counts.index, emotion_counts.values)\n",
    "plt.title('Distribution of Emotions in Dataset')\n",
    "plt.xlabel('Emotions')\n",
    "plt.ylabel('Count')\n",
    "plt.xticks(rotation=45)\n",
    "\n",
    "# Add count labels\n",
    "for bar, count in zip(bars, emotion_counts.values):\n",
    "    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,\n",
    "             str(count), ha='center', va='bottom')\n",
    "\n",
    "# Pie chart\n",
    "plt.subplot(1, 2, 2)\n",
    "plt.pie(emotion_counts.values, labels=emotion_counts.index, autopct='%1.1f%%')\n",
    "plt.title('Emotion Distribution (Percentage)')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "print(\"Emotion distribution:\")\n",
    "for emotion, count in emotion_counts.items():\n",
    "    print(f\"{emotion}: {count} ({count/len(df)*100:.1f}%)\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Visualize Sample Images"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Function to display sample images for each emotion\n",
    "def display_emotion_samples(df, samples_per_emotion=4):\n",
    "    fig, axes = plt.subplots(7, samples_per_emotion, figsize=(15, 20))\n",
    "    \n",
    "    for emotion_idx in range(7):\n",
    "        emotion_data = df[df['emotion'] == emotion_idx]\n",
    "        emotion_name = emotion_labels[emotion_idx]\n",
    "        \n",
    "        for sample_idx in range(samples_per_emotion):\n",
    "            if sample_idx < len(emotion_data):\n",
    "                # Get pixel data and reshape\n",
    "                pixel_data = emotion_data.iloc[sample_idx]['pixels']\n",
    "                pixels = [int(pixel) for pixel in pixel_data.split(' ')]\n",
    "                image = np.array(pixels).reshape(48, 48)\n",
    "                \n",
    "                axes[emotion_idx, sample_idx].imshow(image, cmap='gray')\n",
    "                axes[emotion_idx, sample_idx].set_title(f'{emotion_name}')\n",
    "                axes[emotion_idx, sample_idx].axis('off')\n",
    "    \n",
    "    plt.suptitle('Sample Images for Each Emotion', fontsize=16)\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "\n",
    "display_emotion_samples(df, samples_per_emotion=5)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Pixel Intensity Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Analyze pixel intensity distributions for different emotions\n",
    "def analyze_pixel_intensities(df, sample_size=1000):\n",
    "    plt.figure(figsize=(15, 10))\n",
    "    \n",
    "    for emotion_idx in range(7):\n",
    "        emotion_data = df[df['emotion'] == emotion_idx].sample(n=min(sample_size, len(df[df['emotion'] == emotion_idx])))\n",
    "        emotion_name = emotion_labels[emotion_idx]\n",
    "        \n",
    "        # Calculate mean pixel intensities\n",
    "        mean_intensities = []\n",
    "        for _, row in emotion_data.iterrows():\n",
    "            pixels = [int(pixel) for pixel in row['pixels'].split(' ')]\n",
    "            mean_intensities.append(np.mean(pixels))\n",
    "        \n",
    "        plt.hist(mean_intensities, alpha=0.7, label=emotion_name, bins=30)\n",
    "    \n",
    "    plt.xlabel('Mean Pixel Intensity')\n",
    "    plt.ylabel('Frequency')\n",
    "    plt.title('Distribution of Mean Pixel Intensities by Emotion')\n",
    "    plt.legend()\n",
    "    plt.show()\n",
    "\n",
    "analyze_pixel_intensities(df)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Model Architecture Experiments"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load and preprocess data\n",
    "preprocessor = DataPreprocessor('../data/')\n",
    "X, y = preprocessor.load_fer2013('../data/fer2013.csv')\n",
    "\n",
    "print(f\"Data shape: {X.shape}\")\n",
    "print(f\"Labels shape: {y.shape}\")\n",
    "\n",
    "# Split data\n",
    "X_train, X_val, X_test, y_train, y_val, y_test = preprocessor.split_data(X, y)\n",
    "print(f\"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Compare different model architectures\n",
    "model_builder = EmotionCNN()\n",
    "\n",
    "# Custom CNN\n",
    "custom_model = model_builder.build_custom_cnn()\n",
    "print(\"Custom CNN Architecture:\")\n",
    "custom_model.summary()\n",
    "\n",
    "print(\"\\n\" + \"=\"*50 + \"\\n\")\n",
    "\n",
    "# Transfer learning model\n",
    "transfer_model = model_builder.build_transfer_learning_model('VGG16')\n",
    "print(\"Transfer Learning (VGG16) Architecture:\")\n",
    "transfer_model.summary()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Data Augmentation Visualization"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize data augmentation effects\n",
    "from tensorflow.keras.preprocessing.image import ImageDataGenerator\n",
    "\n",
    "# Take a sample image\n",
    "sample_image = X_train[0:1]  # Shape: (1, 48, 48, 1)\n",
    "\n",
    "# Create data generator with augmentation\n",
    "datagen = ImageDataGenerator(\n",
    "    rotation_range=20,\n",
    "    width_shift_range=0.2,\n",
    "    height_shift_range=0.2,\n",
    "    horizontal_flip=True,\n",
    "    zoom_range=0.2,\n",
    "    fill_mode='nearest'\n",
    ")\n",
    "\n",
    "# Generate augmented images\n",
    "plt.figure(figsize=(15, 5))\n",
    "\n",
    "# Original image\n",
    "plt.subplot(1, 6, 1)\n",
    "plt.imshow(sample_image[0, :, :, 0], cmap='gray')\n",
    "plt.title('Original')\n",
    "plt.axis('off')\n",
    "\n",
    "# Generate 5 augmented versions\n",
    "aug_iter = datagen.flow(sample_image, batch_size=1)\n",
    "for i in range(5):\n",
    "    batch = next(aug_iter)\n",
    "    plt.subplot(1, 6, i + 2)\n",
    "    plt.imshow(batch[0, :, :, 0], cmap='gray')\n",
    "    plt.title(f'Augmented {i+1}')\n",
    "    plt.axis('off')\n",
    "\n",
    "plt.suptitle('Data Augmentation Examples')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Training Set Statistics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Analyze training set statistics\n",
    "print(\"Training Set Statistics:\")\n",
    "print(f\"Total training samples: {len(X_train)}\")\n",
    "print(f\"Image dimensions: {X_train.shape[1:]}\")\n",
    "print(f\"Pixel value range: [{X_train.min():.3f}, {X_train.max():.3f}]\")\n",
    "print(f\"Mean pixel value: {X_train.mean():.3f}\")\n",
    "print(f\"Standard deviation: {X_train.std():.3f}\")\n",
    "\n",
    "# Plot pixel value distribution\n",
    "plt.figure(figsize=(12, 4))\n",
    "\n",
    "plt.subplot(1, 2, 1)\n",
    "plt.hist(X_train.flatten(), bins=50, alpha=0.7)\n",
    "plt.xlabel('Pixel Value')\n",
    "plt.ylabel('Frequency')\n",
    "plt.title('Distribution of Pixel Values')\n",
    "\n",
    "plt.subplot(1, 2, 2)\n",
    "sample_means = [np.mean(img) for img in X_train[:1000]]\n",
    "plt.hist(sample_means, bins=30, alpha=0.7)\n",
    "plt.xlabel('Mean Pixel Value per Image')\n",
    "plt.ylabel('Frequency')\n",
    "plt.title('Distribution of Mean Pixel Values')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "# Class distribution in training set\n",
    "y_train_classes = np.argmax(y_train, axis=1)\n",
    "train_emotion_counts = pd.Series(y_train_classes).value_counts().sort_index()\n",
    "\n",
    "plt.figure(figsize=(10, 6))\n",
    "emotion_names = [emotion_labels[i] for i in train_emotion_counts.index]\n",
    "bars = plt.bar(emotion_names, train_emotion_counts.values)\n",
    "plt.title('Training Set - Emotion Distribution')\n",
    "plt.xlabel('Emotions')\n",
    "plt.ylabel('Count')\n",
    "plt.xticks(rotation=45)\n",
    "\n",
    "# Add percentage labels\n",
    "total_train = len(y_train)\n",
    "for bar, count in zip(bars, train_emotion_counts.values):\n",
    "    percentage = count/total_train*100\n",
    "    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,\n",
    "             f'{count}\\n({percentage:.1f}%)', ha='center', va='bottom')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Model Performance Prediction"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Estimate model performance based on dataset characteristics\n",
    "print(\"Dataset Analysis for Performance Estimation:\")\n",
    "print(\"=\"*50)\n",
    "\n",
    "# Class imbalance analysis\n",
    "class_counts = np.bincount(np.argmax(y, axis=1))\n",
    "max_count = np.max(class_counts)\n",
    "min_count = np.min(class_counts)\n",
    "imbalance_ratio = max_count / min_count\n",
    "\n",
    "print(f\"Class imbalance ratio: {imbalance_ratio:.2f}\")\n",
    "print(f\"Most common class: {emotion_labels[np.argmax(class_counts)]} ({max_count} samples)\")\n",
    "print(f\"Least common class: {emotion_labels[np.argmin(class_counts)]} ({min_count} samples)\")\n",
    "\n",
    "# Expected performance estimation\n",
    "if imbalance_ratio < 2.0:\n",
    "    expected_accuracy = \"70-75%\"\n",
    "elif imbalance_ratio < 5.0:\n",
    "    expected_accuracy = \"65-70%\"\n",
    "else:\n",
    "    expected_accuracy = \"60-65%\"\n",
    "\n",
    "print(f\"\\nExpected model accuracy: {expected_accuracy}\")\n",
    "print(f\"\\nRecommendations:\")\n",
    "if imbalance_ratio > 3.0:\n",
    "    print(\"- Use class weights or data augmentation for minority classes\")\n",
    "    print(\"- Consider focal loss for handling class imbalance\")\n",
    "\n",
    "print(\"- Use data augmentation to increase dataset diversity\")\n",
    "print(\"- Consider transfer learning for better feature extraction\")\n",
    "print(\"- Apply early stopping to prevent overfitting\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.5"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}
